# 08 — Compare against the open DSPy MIPROv2 baseline

**Foundry feature, by contrast:** Foundry's optimizer is a hosted, versioned, **closed preview
product** — a result produced through it isn't independently reproducible by someone without access
to it. `../prompt-agent-optimizer-baselines/_baselines/dspy_mipro/` runs the identical case study
through an open, reproducible optimizer (DSPy's MIPROv2) as a comparison point. **Mode: CLI (real
run costs money; a free dry-run smoke test is also shown).**

In [ ]:
import json, subprocess, sys
from pathlib import Path

# Repo layout: this notebook lives in notebooks/, the pack lives in ../prompt-agent-optimizer-baselines
PACK_ROOT = Path("..").resolve() / "prompt-agent-optimizer-baselines"
AGENT_ID = "01-travel-approval-strict"          # <- the one case study every notebook in this series uses
AGENT_DIR = PACK_ROOT / AGENT_ID

assert AGENT_DIR.exists(), f"Can't find {AGENT_DIR} -- run this notebook from a checkout of the repo."

def run(cmd, cwd=PACK_ROOT):
    """Run a pack CLI tool and print its output, the way you would from a terminal."""
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result

print(f"Pack root : {PACK_ROOT}")
print(f"Case study: {AGENT_ID}")

## What makes this a fair comparison

Same dataset split, same contract-scoring code, same run-manifest shape as the Foundry track — so
its output merges directly into one table:

- `agent_loader.py` reads the identical `agent.yaml` / `instructions.md` / `tools.json` / dataset
  splits / `expected/expectations.json` this pack's Foundry track uses — nothing is re-authored.
- MIPROv2 only ever sees `dataset/optimize.jsonl` (internally re-split into train/val by `--seed`);
  `dataset/holdout.jsonl` is reserved for the final report, mirroring `build_foundry_dataset.py`.
- `metric.py` imports `eval_match`/`validate` directly from `_tools/validate_candidate.py` — not a
  reimplementation of the scoring logic.

## Free dry run (no API key, no cost)

Confirms the whole pipeline wires up correctly for this case study before you spend money on a real
optimization run.

In [ ]:
dspy_dir = PACK_ROOT / "_baselines" / "dspy_mipro"
run(["python3", "run_all.py", "--agents", AGENT_ID, "--seeds", "0", "1", "--dry-run", "--auto", "light"],
    cwd=dspy_dir)

> Requires `pip install -r _baselines/dspy_mipro/requirements.txt` first (see notebook 00). A
> `--dry-run` score is meaningless as evidence of optimization quality — it only proves the wiring
> works, using a zero-cost stub language model in place of a real one.

## Real run (optional, costs money)

```bash
cd prompt-agent-optimizer-baselines/_baselines/dspy_mipro
python run_mipro_baseline.py \
  --agent 01-travel-approval-strict --seed 0 \
  --task-lm openai/gpt-4.1-mini --prompt-lm openai/gpt-5 --auto medium \
  --use-judge --cross-judge
```

`--task-lm` is the model the optimized instructions actually run on; `--prompt-lm` is the (usually
stronger) model MIPROv2 uses to author candidate instructions. This single command produces an
optimized-instructions file, an instruction-level report, and a holdout evaluation — the DSPy-track
equivalent of Steps 03-07 above, in one call — written to
`_baselines/dspy_mipro/results/01-travel-approval-strict/seed0/`.

For a reportable result, run replicate seeds the same way notebook 07 discussed for the Foundry
track (`--seeds 0 1 2 ... 9` via `run_all.py` for k=10).

## Merge both tracks into one comparison table

Reads every DSPy `run_manifest.json` your replicate runs produced, and — if you've saved Foundry run
manifests under `runs/<agent_id>/<run_label>.manifest.json` (notebook 04 writes one automatically) —
merges both into one table with a percentile bootstrap 95% CI per agent per system.

In [ ]:
run(["python3", "compare_to_foundry.py",
     "--results-dir", "results", "--pack-root", "../..", "--out", "results/comparison.json"],
    cwd=dspy_dir)

If you haven't run any real DSPy or Foundry replicates yet, this command says so explicitly rather
than silently leaving half the table blank-looking — that's the expected state the first time you
run this notebook.

## Cross-run textual similarity

How similar is the exported candidate to the original baseline, really — and is that similarity
meaningful, or just shared vocabulary? Two unrelated agents' baselines (`06-sales-brief-underspecified`,
`09-code-review-assistant-strict`) supply the **cross-agent null**: the similarity floor for
genuinely unrelated prompts, which real replicate runs of the *same* agent should clear by a wide
margin. Once you have more than one replicate candidate (notebook 07), add each one to `--texts` to
compare them pairwise instead of just baseline-vs-run1.

In [ ]:
run(["python3", "_tools/similarity_baseline.py",
     "--texts", str(AGENT_DIR / "instructions.md"),
     str(AGENT_DIR / "candidates" / "foundry_run1.md"),
     "--null-texts", str(PACK_ROOT / "06-sales-brief-underspecified" / "instructions.md"),
     str(PACK_ROOT / "09-code-review-assistant-strict" / "instructions.md"),
     "--out", str(AGENT_DIR / "candidates" / "similarity.json")])

In [ ]:
sim = json.loads((AGENT_DIR / "candidates" / "similarity.json").read_text())
print(json.dumps(sim, indent=2))

Read the cosine similarity against **both** nulls the tool reports: the permutation null (what
similarity looks like from shared vocabulary alone, with the same words shuffled) and the cross-agent
null (the floor for genuinely unrelated prompts). A raw cosine number on its own tells you nothing —
it needs a null to be interpreted against.

## Next

Continue to **`09_assemble_final_report.ipynb`** to roll every artifact this series produced into one
promotion decision for the case study agent.